In [1]:
from langchain_core.documents import Document

In [2]:
from langchain_community.document_loaders.text import TextLoader

C:\Users\kkris\AppData\Local\Temp\ipykernel_27540\3315532343.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader
C:\Users\kkris\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
loader = TextLoader("data/python.txt", encoding = "utf-8")
document = loader.load()

In [4]:
#pdf data

#from langchain_community.document_loaders.pdf import PyPDFLoader
#pdf_loader = PyPDFLoader("data/research2.pdf")

In [5]:
#document = pdf_loader.load()

In [6]:
#ingestion pipeline
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            all_docs.extend(doc)
            num_docs += 1
    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [7]:
all_pdf_documents = load_all_pdfs()

total pdfs: 1
total pages: 21


In [8]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

In [9]:
!pip install langchain_text_splitters

Defaulting to user installation because normal site-packages is not writeable


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size = 500, chunk_overlap = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs


chunks = split_docs(all_pdf_documents)

In [11]:
# Embedding manager

from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    def __init__(self, model_name = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("loading_model", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions = ", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self,text):
        embeddings = self.model.encode(text, show_progress_bar = True)
        print("embedding shape:", embeddings.shape)
        return embeddings

In [12]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNINGS"] = "1"

In [13]:
embedding_manager = EmbeddingManager()

loading_model all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3628.87it/s]


embedding dimensions =  384


C:\Users\kkris\AppData\Local\Temp\ipykernel_27540\4105255923.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions = ", self.model.get_sentence_embedding_dimension())


In [14]:
import chromadb
import uuid

In [15]:
#VectorStoreManager 
class VectorStoreManager:
    def __init__(self,persist_directory = "data/vector_store", collection_name = "pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        
        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok = True)

        #create a client
        self.client = chromadb.PersistentClient(path = self.persist_directory)

        #create the collection
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description":"vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents)!= len(embeddings):
            raise ValueError("num of documents docs not match num of embeddings")

        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids = ids,
                metadatas = all_metadata,
                documents = documents_content,
                embeddings = embeddings_list
            )

        print("total document added in vector store = ", len(documents_content))
        print("docs in collection:", self.collection.count())

        
        

In [16]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 488


In [17]:
texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches: 100%|██████████| 8/8 [00:29<00:00,  3.70s/it]


embedding shape: (244, 384)
total document added in vector store =  244
docs in collection: 732


In [18]:
#Retriever pipeline

class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k = 5, score_threshold = 0.0):
        #query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        #semantic search
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        #cosine simliarity
        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1-distance 

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id":doc_id,
                        "document":document,
                        "metadata":metadata,
                        "distance":distance,
                        "similarity_score":similarity_score,
                        "rank": i+1
                    })

        else:
            print("no documents found")

        return retrieved_docs
        


In [19]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)
rag_retriever.retrieve("What is RAG")

Batches: 100%|██████████| 1/1 [00:00<00:00, 29.73it/s]

embedding shape: (1, 384)


[{'id': 'doca3d29c08-113b-4b7e-9337-96475d625ded',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'doc_index': 11,
   'moddate': '2024-03-28T00:54:45+00:00',
   'page': 0,
   'creator': 'LaTeX with hyperref',
   'producer': 'pdfTeX-1.40.25',
   'source': 'data/pdfs\\research2.pdf',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'content_length': 288,
   'trapped': '/False',
   'author': '',
   'keywords': '',
   'title': '',
   'page_label': '1',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'total_pages': 21,
   'subject': ''},
  'distance': 0.4226757287979126,
  'similarity_score': 0.5773242712020874,
  'rank': 1},
 {'id': 'doc314ee337-62

In [ ]:
API_KEY_GROQ = "API-Key"

In [27]:
!pip install langchain-groq

Defaulting to user installation because normal site-packages is not writeable


In [28]:
from langchain_groq import ChatGroq

In [32]:
llm = ChatGroq(
    groq_api_key = API_KEY_GROQ,
    model = "qwen/qwen3.6-27b",
    temperature = 0.1,
    max_tokens = 1024
)

In [41]:
#generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k = 3):
    results = retriever.retrieve(query,top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""
    if not context:
        print("we found no relevent context for the given query")

    #context + query
    prompt = f""" use given context to generate the answer for the query
                  Context:{context}
                  Query:{query}"""

    response = llm.invoke([prompt.format(ocntext = context, query = query)])
    return response.content

In [42]:
answer = generate_output("What is RAG", rag_retriever, llm)


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.55it/s]


embedding shape: (1, 384)


In [43]:
print(answer)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Context:** The provided text is repetitive and seems to be an excerpt from an academic paper (arXiv:2312.10997v5, cs.CL, 27 Mar 2024). It mentions: "In this survey, we present a thorough and systematic review of the state-of-the-art RAG methods, delineating its evolution through paradigms including naive RAG, and speculate on upcoming trends and innovations."
   - **Query:** "What is RAG"
   - **Task:** Use the given context to generate an answer for the query.

2.  **Identify Key Information in Context:**
   - The context explicitly mentions "RAG methods" and "naive RAG".
   - It describes RAG as having an "evolution through paradigms including naive RAG".
   - It's part of a survey reviewing "state-of-the-art RAG methods".
   - *Crucially*, the context does *not* explicitly define what RAG stands for or what it is. It only mentions it as a subject of a survey with paradigms like "naive RAG".

3.  **Determine Cons